# BigMart Dataset Cleaning

**Problem:** Clean missing, inconsistent, and unusual values in the BigMart training data.

Explore the data, identify issues, clean values that can be supported by the available information, review unusual observations, and save a cleaned copy.

## Exploratory Phase

Review the records, column types, distributions, missing values, duplicates, and category labels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('bigmart_train - bigmart_train.csv')
print('Dataset shape:', df.shape)

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values by column:')
display(df.isna().sum().rename('missing').to_frame())
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
print('Fat-content labels:')
display(df['Item_Fat_Content'].value_counts())
print('Outlet sizes, including missing:')
display(df['Outlet_Size'].value_counts(dropna=False))
print('Rows with zero item visibility:', (df['Item_Visibility'] == 0).sum())

## Check for Outliers

Use boxplots and visible cutoffs to flag unusual values for review. These cutoffs do not prove that a value is an error.

In [ ]:
visibility_limit = 0.20
sales_limit = 6500

high_visibility = df[df['Item_Visibility'] > visibility_limit]
high_sales = df[df['Item_Outlet_Sales'] > sales_limit]

print('Rows with visibility above', visibility_limit, ':', len(high_visibility))
print('Rows with sales above', sales_limit, ':', len(high_sales))

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(df['Item_Weight'].dropna())
plt.title('Item Weight')
plt.ylabel('Weight')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(df['Item_Visibility'])
plt.title('Item Visibility')
plt.ylabel('Visibility')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(df['Item_MRP'])
plt.title('Item MRP')
plt.ylabel('Price')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(df['Item_Outlet_Sales'])
plt.title('Item Outlet Sales')
plt.ylabel('Sales')
plt.show()

## Address the Findings

Standardize category labels and fill missing values using item or outlet patterns, with an overall median as a fallback where needed.

In [ ]:
df_clean = df.drop_duplicates().copy()
df_clean['Item_Fat_Content'] = df_clean['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})
df_clean['Item_Fat_Content'].value_counts()

In [ ]:
# Use the median weight for the same item; use the dataset median for any remaining gaps.
item_weight_median = df_clean.groupby('Item_Identifier')['Item_Weight'].transform('median')
df_clean['Item_Weight'] = df_clean['Item_Weight'].fillna(item_weight_median)
df_clean['Item_Weight'] = df_clean['Item_Weight'].fillna(df_clean['Item_Weight'].median())
print('Missing item weights:', df_clean['Item_Weight'].isna().sum())

In [ ]:
df.groupby('Outlet_Type')['Outlet_Size'].value_counts().unstack(fill_value=0)

In [ ]:
outlet_size_by_type = {
    'Grocery Store': 'Small',
    'Supermarket Type1': 'Small',
    'Supermarket Type2': 'Medium',
    'Supermarket Type3': 'Medium'
}

In [ ]:
missing_size = df_clean['Outlet_Size'].isna()
df_clean.loc[missing_size, 'Outlet_Size'] = (
    df_clean.loc[missing_size, 'Outlet_Type'].map(outlet_size_by_type)
)
print('Missing outlet sizes:', df_clean['Outlet_Size'].isna().sum())

In [ ]:
df_clean.loc[df_clean['Item_Visibility'] == 0, 'Item_Visibility'] = pd.NA
print('Missing visibility values after replacing zero:', df_clean['Item_Visibility'].isna().sum())

In [ ]:
item_visibility_median = df_clean.groupby('Item_Identifier')['Item_Visibility'].transform('median')

In [ ]:
df_clean['Item_Visibility'] = df_clean['Item_Visibility'].fillna(item_visibility_median)
print('Missing visibility values:', df_clean['Item_Visibility'].isna().sum())

In [ ]:
df_clean['Item_Visibility'] = df_clean['Item_Visibility'].fillna(
    df_clean['Item_Visibility'].median()
)
print('Missing visibility values:', df_clean['Item_Visibility'].isna().sum())

## Review the Cleaned Data

Check the cleaned table for remaining missing values and duplicates, compare key distributions, then save the result.

In [ ]:
print('Original shape:', df.shape)
print('Cleaned shape:', df_clean.shape)
print('Duplicate rows:', df_clean.duplicated().sum())
print('Missing values by column:')
display(df_clean.isna().sum().rename('missing').to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].boxplot([df['Item_Visibility'], df_clean['Item_Visibility']])
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['Before', 'After'])
axes[0].set_title('Item Visibility')
axes[0].set_ylabel('Visibility')

axes[1].boxplot([df['Item_Outlet_Sales'], df_clean['Item_Outlet_Sales']])
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['Before', 'After'])
axes[1].axhline(sales_limit, color='orange', linestyle='--', label='Review cutoff: 6,500')
axes[1].set_title('Item Outlet Sales')
axes[1].set_ylabel('Sales')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
output_path = 'bigmart_train_cleaned.csv'
df_clean.to_csv(output_path, index=False)
print('Cleaned dataset saved to:', output_path)

## Conclusion

The notebook standardized fat-content labels, filled missing item weights and outlet sizes, and replaced zero or missing item visibility with item-based medians where possible. The source CSV remains unchanged. Simple cutoffs are documented for review, while the sales observations are kept because unusually high values may be genuine.